# FUSE — manufacturing preparation

This notebook turns the **human-approved Stage 4 fragment** into checked manufacturing artifacts. It keeps the workflow interactive while using the tested `fuse_manufacturing` backend for geometry operations and FreeCAD export.

The notebook will:

1. locate and inspect the selected fragment hypothesis;
2. visualize the source mesh;
3. convert the arbitrary reconstruction scale to millimetres;
4. construct or validate the mating interface;
5. check watertightness, winding, connected components and volume;
6. preview the fragment in object and print coordinates;
7. create `.FCStd`, STL, 3MF and optional STEP/BREP files.

> **Physical gate:** do not set `SCALE_CONFIRMED = True` until the scale has been derived from a real measurement. A planar cap is suitable for the current Nessie prototype, but it is not a fracture-exact mating surface.

In [1]:
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import trimesh
import yaml
from IPython.display import JSON, Markdown, display

from fuse_manufacturing.cli import (
    apply_scale,
    discover_fragment,
    enforce_checks,
    physical_scale,
    resolve_file,
    resolve_handoff,
    run_manufacturing,
)
from fuse_manufacturing.mesh_ops import (
    cap_planar,
    clean_mesh,
    closed_mesh_interface,
    load_triangle_mesh,
    mesh_health,
    orient_for_print,
    stitch_measured_patch,
)

print('Manufacturing environment loaded.')

Manufacturing environment loaded.


## 1. Configuration

Edit only this cell for a normal run. `HANDOFF_RUN = "latest"` selects the newest Streamlit/verification hand-off. For reproducibility, replace it later with a specific path such as `04_verification/runs/handoff_20260810T...`.

In [2]:
# Shared FUSE paths
DATA_ROOT = Path('/workspace/data')
HANDOFF_RUN = 'latest'
FRAGMENT_MESH = None          # None = auto-discover in the hand-off
BROKEN_OBJECT_MESH = None     # optional mesh for proximity checks

# Physical scale — choose ONE method. Values below are intentionally unset.
SOURCE_TO_MM = None
KNOWN_DISTANCE_SOURCE_UNITS = None
KNOWN_DISTANCE_MM = None
SCALE_CONFIRMED = False

# Mating interface
INTERFACE_METHOD = 'planar_cap'   # planar_cap | measured_patch | closed_mesh
MEASURED_PATCH_MESH = None
CLEARANCE_MM = 0.20
MEASURED_PATCH_NORMAL_SIGN = 1   # change to -1 if the offset goes outward
MAX_PLANARITY_RMS_MM = 0.35

# Validation and export
MAX_BREP_FACES = 200_000
FREECAD_MESH_TOLERANCE_MM = 0.03
RUN_FREECAD = True

# Conservative initial print settings; review these in the actual slicer.
PRINT_MATERIAL = 'PLA'
NOZZLE_MM = 0.40
LAYER_HEIGHT_MM = 0.12
PERIMETERS = 4
INFILL_PERCENT = 100

CONFIG_PATH = DATA_ROOT / '05_manufacturing' / 'manufacturing-notebook.yaml'

## 2. Visualization helpers

In [3]:
def mesh_trace(mesh, name, color, opacity=1.0):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name, color=color, opacity=opacity, flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.75, specular=0.20, roughness=0.65),
    )


def show_meshes(items, title, interface=None):
    fig = go.Figure()
    for mesh, name, color, opacity in items:
        fig.add_trace(mesh_trace(mesh, name, color, opacity))
    if interface is not None:
        point = np.asarray(interface.point_mm)
        normal = np.asarray(interface.normal_mm)
        length = max(float(np.ptp(np.vstack([m.bounds for m, *_ in items]), axis=0).max()) * 0.20, 1.0)
        end = point + normal * length
        fig.add_trace(go.Scatter3d(
            x=[point[0], end[0]], y=[point[1], end[1]], z=[point[2], end[2]],
            mode='lines+markers', name='interface normal',
            line=dict(color='#ef4444', width=7), marker=dict(size=4),
        ))
    fig.update_layout(
        title=title, height=720, margin=dict(l=0, r=0, t=55, b=0),
        scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
        legend=dict(orientation='h'),
    )
    fig.show()


def show_json(value):
    display(JSON(value, expanded=True))

## 3. Locate and inspect the approved fragment

This stage does not rerun Kaolin detection. It consumes only the component(s) accepted in the Streamlit candidate-review step.

In [4]:
handoff_dir = resolve_handoff(DATA_ROOT, HANDOFF_RUN)
manifest_path = handoff_dir / 'handoff_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing Stage 4 decision manifest: {manifest_path}')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
fragment_path = discover_fragment(handoff_dir, FRAGMENT_MESH, DATA_ROOT)
source_mesh = load_triangle_mesh(fragment_path)
source_health = mesh_health(source_mesh)

print('Hand-off:', handoff_dir)
print('Fragment:', fragment_path)
print('Selected component IDs:', manifest.get('selected_component_ids', []))
show_json({'manifest': manifest, 'source_unit_health': source_health})

FileNotFoundError: No Stage 4 hand-off run exists under /workspace/data/04_verification/runs

In [ ]:
show_meshes(
    [(source_mesh, 'approved fragment — source coordinates', '#2563eb', 1.0)],
    'Stage 4 fragment hypothesis (arbitrary reconstruction units)',
)

## 4. Resolve physical scale

VGGT coordinates do not inherently mean millimetres. This cell deliberately stops until you enter and confirm a measured scale in the configuration cell.

In [ ]:
units_config = {
    'source_to_mm': SOURCE_TO_MM,
    'known_distance_source_units': KNOWN_DISTANCE_SOURCE_UNITS,
    'known_distance_mm': KNOWN_DISTANCE_MM,
    'confirmed': SCALE_CONFIRMED,
}
scale_to_mm = physical_scale(units_config)
scaled_exterior = apply_scale(source_mesh, scale_to_mm)

print(f'Scale: 1 source unit = {scale_to_mm:.6g} mm')
print('Fragment extents [mm]:', np.round(scaled_exterior.extents, 3))

## 5. Construct or validate the mating interface

- `planar_cap` closes one approximately planar boundary loop: useful for the current prototype.
- `measured_patch` stitches an offset scan of the real fracture face: intended accurate route.
- `closed_mesh` accepts an already watertight fragment.

In [ ]:
measured_patch_mm = None
if INTERFACE_METHOD == 'planar_cap':
    final_mesh, interface = cap_planar(
        scaled_exterior, max_planarity_rms_mm=MAX_PLANARITY_RMS_MM
    )
elif INTERFACE_METHOD == 'measured_patch':
    patch_path = resolve_file(MEASURED_PATCH_MESH, DATA_ROOT, handoff_dir)
    if patch_path is None:
        raise ValueError('MEASURED_PATCH_MESH is required for measured_patch')
    patch_source = apply_scale(load_triangle_mesh(patch_path), scale_to_mm)
    final_mesh, interface, measured_patch_mm = stitch_measured_patch(
        scaled_exterior, patch_source,
        clearance_mm=CLEARANCE_MM,
        normal_sign=MEASURED_PATCH_NORMAL_SIGN,
    )
elif INTERFACE_METHOD == 'closed_mesh':
    final_mesh = clean_mesh(scaled_exterior)
    interface = closed_mesh_interface(final_mesh)
else:
    raise ValueError(f'Unsupported INTERFACE_METHOD={INTERFACE_METHOD!r}')

final_mesh = clean_mesh(final_mesh)
show_json(interface.as_dict())

In [ ]:
show_meshes(
    [
        (scaled_exterior, 'open exterior', '#2563eb', 0.34),
        (final_mesh, 'closed manufacturing mesh', '#f59e0b', 0.88),
    ],
    'Mating-interface construction',
    interface=interface,
)

## 6. Run digital manufacturing gates

The export cell is allowed to continue only when the required geometry checks pass.

In [ ]:
checks_config = {
    'require_single_component': True,
    'require_watertight': True,
    'require_winding_consistent': True,
    'max_brep_faces': MAX_BREP_FACES,
    'freecad_mesh_tolerance_mm': FREECAD_MESH_TOLERANCE_MM,
}
health = mesh_health(final_mesh)
failures = enforce_checks(health, checks_config)
show_json(health)
if failures:
    raise ValueError('Manufacturing gate failed: ' + '; '.join(failures))
print('All required digital geometry gates passed.')

## 7. Preview print orientation

The interface is rotated toward the build plate and the lowest point is translated to `Z = 0`. This is a starting orientation, not a substitute for slicer support analysis.

In [ ]:
print_mesh, print_transform = orient_for_print(final_mesh, interface)
print('Print bounds [mm]:')
print(np.round(print_mesh.bounds, 3))
print('Object-to-print transform:')
print(np.array2string(print_transform, precision=5, suppress_small=True))

show_meshes(
    [(print_mesh, 'print-oriented fragment', '#10b981', 1.0)],
    'Print orientation — build plate at Z = 0',
)

## 8. Save the resolved configuration and export

`RUN_FREECAD = True` creates the `.FCStd` document and attempts STEP/BREP conversion. Set it to `False` only when debugging the mesh pipeline without invoking FreeCAD. Every run receives a new timestamped output directory.

In [ ]:
config = {
    'input': {
        'handoff_run': str(handoff_dir),
        'fragment_mesh': str(fragment_path),
        'broken_object_mesh': BROKEN_OBJECT_MESH,
    },
    'units': units_config,
    'interface': {
        'method': INTERFACE_METHOD,
        'measured_patch_mesh': MEASURED_PATCH_MESH,
        'clearance_mm': CLEARANCE_MM,
        'measured_patch_normal_sign': MEASURED_PATCH_NORMAL_SIGN,
        'max_planarity_rms_mm': MAX_PLANARITY_RMS_MM,
    },
    'checks': {
        **checks_config,
        'proximity_warning_mm': 0.10,
    },
    'print': {
        'orientation': 'interface_normal_to_bed',
        'material': PRINT_MATERIAL,
        'nozzle_mm': NOZZLE_MM,
        'layer_height_mm': LAYER_HEIGHT_MM,
        'perimeters': PERIMETERS,
        'infill_percent': INFILL_PERCENT,
        'supports': 'auto',
    },
    'output': {'run_dir': None},
}
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Resolved configuration:', CONFIG_PATH)
display(Markdown('```yaml\n' + CONFIG_PATH.read_text(encoding='utf-8') + '\n```'))

In [ ]:
run_dir = run_manufacturing(
    CONFIG_PATH,
    DATA_ROOT,
    skip_freecad=not RUN_FREECAD,
)
print('Manufacturing run complete:', run_dir)

## 9. Review outputs and report

The final decision gates that remain outside this notebook are a physical dry-fit and review in the slicer configured for the actual printer, nozzle and material.

In [ ]:
report_path = run_dir / 'manufacturing_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
show_json(report)

print('\nCreated files:')
for path in sorted(run_dir.iterdir()):
    if path.is_file():
        print(f'  {path.name:38s} {path.stat().st_size / 1024:10.1f} KiB')

In [ ]:
exported_model = load_triangle_mesh(run_dir / 'repair_model_mm.stl')
exported_print = load_triangle_mesh(run_dir / 'repair_print_oriented.stl')
show_meshes(
    [
        (exported_model, 'object frame', '#2563eb', 0.45),
        (exported_print, 'print frame', '#10b981', 0.72),
    ],
    'Exported geometry — object frame and print frame',
)

### Main artifacts

| File | Purpose |
|---|---|
| `repair.FCStd` | FreeCAD document with object-frame and print-oriented meshes |
| `repair_model_mm.stl` | Watertight fragment in the aligned FUSE/VGGT frame |
| `repair_print_oriented.stl` | Fragment placed on `Z = 0` for slicing |
| `repair_print_oriented.3mf` | Same print mesh with millimetre units embedded |
| `repair.step` / `repair.brep` | Editable solid when mesh-to-BREP conversion succeeds |
| `manufacturing_report.json` | Scale, geometry gates, transforms, limits and hashes |
| `fuse_print_bundle.zip` | Portable bundle for FreeCAD/slicer verification |

No generic G-code is generated because it must be produced using the exact printer profile.